# yatayat-vision — vehicle detector training (Module 1)

Fine-tunes YOLO11n on a Nepal-relevant subset of BMD-45 (see `docs/data-cards/bmd45.md` in the repo). This runs on Colab because the local machine this project is built on has no GPU.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

Storage is split deliberately:
- **Drive** (`/content/drive/MyDrive/yatayat-vision/`) holds only what's expensive to regenerate and small enough to fit a free account: training checkpoints, exported weights, and a small manifest recording exactly what dataset was used.
- **Colab's local disk** holds the dataset subset itself (images + labels). BMD-45's source images are large enough that persisting the full subset to Drive across sessions was actually exhausting a free Drive account's quota - see `docs/data-cards/bmd45.md`. Rebuilding it each session costs a few minutes, which is cheap by comparison.

Re-running this notebook in a later session reuses whatever's on Drive (checkpoints resume, don't restart) and rebuilds the dataset subset fresh.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/yatayat-vision'
RUNS_DIR = f'{DRIVE_ROOT}/runs'
EXPORTS_DIR = f'{DRIVE_ROOT}/exports'
MANIFEST_BACKUP_DIR = f'{DRIVE_ROOT}/dataset_manifests'  # small bookkeeping copies only - see below

# The dataset itself lives on Colab's local disk, not Drive - see the "Build
# the dataset subset" section for why.
DATA_DIR = '/content/bmd45_subset'

import os
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(EXPORTS_DIR, exist_ok=True)
os.makedirs(MANIFEST_BACKUP_DIR, exist_ok=True)

In [ ]:
!pip install -q ultralytics huggingface_hub Pillow

# Belt-and-suspenders against the Xet rate-limit issue: some Colab images
# ship hf_xet preinstalled, and relying only on the HF_HUB_DISABLE_XET env var
# has proven unreliable here across huggingface_hub versions. Uninstalling
# the package outright forces huggingface_hub onto the plain HTTP downloader
# unconditionally, regardless of any env-var/version quirk.
!pip uninstall -y -q hf_xet 2>/dev/null || true

In [ ]:
import os

# Optional but recommended: Hugging Face's Xet transfer protocol rate-limits
# unauthenticated requests hard, and Colab's shared IP pool trips it fast (a
# 429 on the very first download is the symptom). An authenticated request
# gets a much higher limit. Add a token as a Colab secret named HF_TOKEN (key
# icon in the left sidebar) - free to create at huggingface.co/settings/tokens,
# read-only scope is enough. Skipped silently if no such secret exists; the
# retry/backoff in the dataset script below still helps either way.
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('using HF_TOKEN from Colab secrets')
except Exception:
    print('no HF_TOKEN secret found - continuing without one (lower rate limits, relies on retry/backoff)')

## Build the dataset subset

This is the exact same subsetting/conversion logic as `cv-service/scripts/bmd45_subset_and_convert.py` in the repo (kept in sync manually — see that file, and `docs/data-cards/bmd45.md`, for the documented rationale). It downloads only the annotation JSON plus the selected images from Hugging Face — never the full ~153GB dataset — and re-encodes each image as JPEG (quality 90) since BMD-45's source PNGs run ~3.5MB each, which at subset scale is enough to exhaust a free Google Drive account.

The dataset itself is built onto Colab's **local** disk (`/content/bmd45_subset`), not Drive — it's cheap to rebuild each session (a few minutes) and there's no reason to spend Drive's limited free quota on data that regenerates deterministically from the same seed. Only training checkpoints, exported weights, and a small manifest get persisted to Drive.

In [ ]:
%%writefile bmd45_subset_and_convert.py
"""Build a Nepal-relevant subset of BMD-45 and convert it to YOLO format.
See docs/data-cards/bmd45.md in the yatayat-vision repo for the rationale.
Kept identical to cv-service/scripts/bmd45_subset_and_convert.py.
"""

import argparse
import json
import os
import random
import shutil
import time

# Must be set before huggingface_hub is imported - read once into a
# module-level constant at import time. See the HF_TOKEN cell above for why.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from huggingface_hub import hf_hub_download
from PIL import Image

REPO_ID = "iisc-aim/BMD-45"

CATEGORY_MERGE = {
    "Two-wheeler": "two_wheeler",
    "Hatchback": "car", "Sedan": "car", "SUV": "car", "MUV": "car", "Van": "car",
    "Three-wheeler": "three_wheeler",
    "Bus": "bus", "Mini-bus": "bus", "Tempo-traveller": "bus",
    "Truck": "truck", "LCV": "truck",
    "Bicycle": "bicycle",
}
CLASS_NAMES = ["two_wheeler", "car", "three_wheeler", "bus", "truck", "bicycle"]
CLASS_IDS = {name: i for i, name in enumerate(CLASS_NAMES)}
BUCKET_ORDER = ["bicycle", "bus", "truck", "three_wheeler", "car", "two_wheeler"]

SPLITS = {
    "train": {"repo_dir": "BMD-45-Train", "budget": 4000, "per_class_target": 900},
    "val": {"repo_dir": "BMD-45-Val", "budget": 800, "per_class_target": 200},
}


def _with_retry(fn, *args, retries=6, base_delay=5.0, **kwargs):
    """Retries transient failures (rate limits, flaky Colab networking) with
    exponential backoff. Hugging Face's 429s are usually gone within a minute."""
    for attempt in range(retries):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            if attempt == retries - 1:
                raise
            delay = base_delay * (2 ** attempt)
            print(f"  download failed ({exc}); retrying in {delay:.0f}s ({attempt + 1}/{retries})")
            time.sleep(delay)


def download_annotations(repo_dir, cache_dir):
    path = _with_retry(hf_hub_download, REPO_ID, f"{repo_dir}/_annotations.coco.json", repo_type="dataset", local_dir=str(cache_dir))
    with open(path) as f:
        return json.load(f)


def select_images(coco, budget, per_class_target, seed):
    id2name = {c["id"]: c["name"] for c in coco["categories"]}
    bucket_images = {name: set() for name in CLASS_NAMES}
    for ann in coco["annotations"]:
        if ann.get("iscrowd"):
            continue
        bucket_images[CATEGORY_MERGE[id2name[ann["category_id"]]]].add(ann["image_id"])

    rng = random.Random(seed)
    selected = set()
    for bucket in BUCKET_ORDER:
        if len(selected) >= budget:
            break
        already = len(bucket_images[bucket] & selected)
        need = max(0, per_class_target - already)
        if need == 0:
            continue
        pool = list(bucket_images[bucket] - selected)
        rng.shuffle(pool)
        for image_id in pool[:need]:
            if len(selected) >= budget:
                break
            selected.add(image_id)
    return selected


def coco_bbox_to_yolo(bbox, img_w, img_h):
    x_min, y_min, w, h = bbox
    x_center = (x_min + w / 2) / img_w
    y_center = (y_min + h / 2) / img_h
    clamp = lambda v: min(max(v, 0.0), 1.0)
    return clamp(x_center), clamp(y_center), clamp(w / img_w), clamp(h / img_h)


def build_split(split_name, cfg, out_dir, seed, workers):
    cache_dir = out_dir / "_hf_cache"
    coco = download_annotations(cfg["repo_dir"], cache_dir)
    id2name = {c["id"]: c["name"] for c in coco["categories"]}
    selected_ids = select_images(coco, cfg["budget"], cfg["per_class_target"], seed)

    images_by_id = {im["id"]: im for im in coco["images"] if im["id"] in selected_ids}
    anns_by_image = {i: [] for i in images_by_id}
    for ann in coco["annotations"]:
        if ann.get("iscrowd") or ann["image_id"] not in images_by_id:
            continue
        anns_by_image[ann["image_id"]].append(ann)

    images_dir = out_dir / split_name / "images"
    labels_dir = out_dir / split_name / "labels"
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    def flat_stem(file_name):
        rel = Path(file_name)
        return f"{rel.parent.name}__{rel.stem}"

    def fetch_one(image_id):
        info = images_by_id[image_id]
        dest = images_dir / f"{flat_stem(info['file_name'])}.jpg"
        if dest.exists():
            return image_id, None
        try:
            src = _with_retry(hf_hub_download, REPO_ID, f"{cfg['repo_dir']}/{info['file_name']}", repo_type="dataset", local_dir=str(cache_dir), retries=4, base_delay=3.0)
        except Exception as exc:
            return image_id, str(exc)
        try:
            with Image.open(src) as im:
                im.convert("RGB").save(dest, "JPEG", quality=90)
        except Exception as exc:
            return image_id, f"failed to re-encode as jpeg: {exc}"
        finally:
            Path(src).unlink(missing_ok=True)
        return image_id, None

    failures = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(fetch_one, iid): iid for iid in images_by_id}
        for fut in as_completed(futures):
            image_id, err = fut.result()
            if err:
                failures.append((image_id, err))
    failed_ids = {image_id for image_id, _ in failures}

    bucket_counts = {name: 0 for name in CLASS_NAMES}
    for image_id, info in images_by_id.items():
        if image_id in failed_ids:
            continue
        lines = []
        seen_buckets = set()
        for ann in anns_by_image[image_id]:
            bucket = CATEGORY_MERGE[id2name[ann["category_id"]]]
            x, y, w, h = coco_bbox_to_yolo(ann["bbox"], info["width"], info["height"])
            if w <= 0 or h <= 0:
                continue
            lines.append(f"{CLASS_IDS[bucket]} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
            seen_buckets.add(bucket)
        stem = flat_stem(info["file_name"])
        label_path = labels_dir / f"{stem}.txt"
        if not label_path.exists():
            label_path.write_text("\n".join(lines) + ("\n" if lines else ""))
        for b in seen_buckets:
            bucket_counts[b] += 1

    return {"split": split_name, "requested_budget": cfg["budget"], "images_selected": len(images_by_id) - len(failed_ids), "images_failed": len(failed_ids), "per_class_image_counts": bucket_counts}


def write_data_yaml(out_dir):
    names = "\n".join(f"  {i}: {name}" for i, name in enumerate(CLASS_NAMES))
    (out_dir / "data.yaml").write_text(f"path: {out_dir}\ntrain: train/images\nval: val/images\nnames:\n{names}\n")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", default="data/bmd45_subset")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--workers", type=int, default=8)
    parser.add_argument("--scale", type=float, default=1.0)
    args = parser.parse_args()

    out_dir = Path(args.out).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    manifest = {"source": REPO_ID, "seed": args.seed, "scale": args.scale, "class_names": CLASS_NAMES, "splits": []}
    for split_name, cfg in SPLITS.items():
        scaled_cfg = {"repo_dir": cfg["repo_dir"], "budget": max(1, int(cfg["budget"] * args.scale)), "per_class_target": max(1, int(cfg["per_class_target"] * args.scale))}
        print(f"building {split_name}: budget={scaled_cfg['budget']} per_class_target={scaled_cfg['per_class_target']}")
        result = build_split(split_name, scaled_cfg, out_dir, args.seed, args.workers)
        print(result)
        manifest["splits"].append(result)

    write_data_yaml(out_dir)
    (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print(f"done. dataset at {out_dir}")


if __name__ == "__main__":
    main()


In [ ]:
import os

if os.path.exists(f'{DATA_DIR}/data.yaml'):
    print('subset already built this session, reusing it')
else:
    # Full subset: ~4000 train + ~800 val images, not the full ~153GB dataset.
    # Re-running with the same seed (default 42) reproduces the identical subset.
    # Lives on local disk (see the section above) - rebuilds fresh each session.
    !python bmd45_subset_and_convert.py --out {DATA_DIR} --workers 12

!cat {DATA_DIR}/manifest.json

# Small, durable record of what this training run actually used - the bulky
# images/labels stay local-only, but this bookkeeping copy is worth keeping.
import shutil
shutil.copy2(f'{DATA_DIR}/manifest.json', f'{MANIFEST_BACKUP_DIR}/manifest.json')
shutil.copy2(f'{DATA_DIR}/data.yaml', f'{MANIFEST_BACKUP_DIR}/data.yaml')

## Train

Transfer learning from COCO-pretrained YOLO11n, a short fixed schedule (no hyperparameter search — a free Colab session doesn't have the GPU-hours budget for that). Ultralytics checkpoints every epoch by default; `resume=True` on a second run picks back up from `last.pt` in the same Drive-persisted project directory if a session gets cut off partway through.

In [ ]:
from ultralytics import YOLO

RUN_NAME = 'vehicle_detector'
last_checkpoint = f'{RUNS_DIR}/{RUN_NAME}/weights/last.pt'

if os.path.exists(last_checkpoint):
    print('resuming from a previous checkpoint on Drive')
    model = YOLO(last_checkpoint)
    results = model.train(resume=True)
else:
    model = YOLO('yolo11n.pt')
    results = model.train(
        data=f'{DATA_DIR}/data.yaml',
        epochs=50,
        imgsz=640,
        batch=16,
        project=RUNS_DIR,
        name=RUN_NAME,
        exist_ok=True,
    )

## Validate

mAP on the held-out val split (the ~800-image subset the training run never saw), per class and overall.

In [ ]:
best = YOLO(f'{RUNS_DIR}/{RUN_NAME}/weights/best.pt')
metrics = best.val(data=f'{DATA_DIR}/data.yaml')
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)
print('per-class mAP50-95:', dict(zip(metrics.names.values(), metrics.box.maps)))

## Export for local (M1) inference

ONNX for portability, CoreML for native Apple Silicon acceleration. Both get saved to Drive — download them from there and place them under `cv-service/data/models/` locally (gitignored, same as everything else under `data/`) to run the local smoke test.

In [ ]:
import shutil

onnx_path = best.export(format='onnx')
coreml_path = best.export(format='coreml')

for path in [onnx_path, coreml_path]:
    dest = f'{EXPORTS_DIR}/{os.path.basename(path)}'
    if os.path.isdir(path):
        shutil.copytree(path, dest, dirs_exist_ok=True)
    else:
        shutil.copy2(path, dest)
    print('saved to', dest)